In [3]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# Load cleaned data
df = pd.read_csv("appointments_clean.csv")

# Target
y = df["no_show"]

# Features (exclude identifiers, dates and target)
features = [
    "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "lead_days", "clinic_id"
]

X = df[features]

# Separate numeric and categorical columns
numeric_features = [
    "age", "distance_km", "fee_inr",
    "prior_no_shows", "lead_days"
]
categorical_features = [
    "gender", "booking_channel", "visit_type",
    "reminder_sent", "clinic_id"
]

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Export test risk scores
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "actual_no_show": y_test.values,
    "risk_score": y_prob
})

risk_scores["risk_band"] = pd.cut(
    risk_scores["risk_score"],
    bins=[-0.001, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

risk_scores.to_csv("test_risk_scores.csv", index=False)

print("\nExported test_risk_scores.csv")
print(risk_scores.head())
print("Test records:", len(risk_scores))


Accuracy: 0.652
Precision: 0.353
Recall: 0.682
F1 Score: 0.465
ROC-AUC: 0.715

Confusion Matrix:
 [[99 55]
 [14 30]]

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.64      0.74       154
           1       0.35      0.68      0.47        44

    accuracy                           0.65       198
   macro avg       0.61      0.66      0.60       198
weighted avg       0.76      0.65      0.68       198


Exported test_risk_scores.csv
  appointment_id  actual_no_show  risk_score risk_band
0          A0856               0    0.118547       Low
1          A0942               0    0.256463       Low
2          A0163               0    0.680039      High
3          A0663               0    0.133233       Low
4          A0854               0    0.290593       Low
Test records: 198
